In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms, models
import random

# Emotion labels
EMOTIONS = {
    0: "Neutral", 1: "Happy", 2: "Sad", 3: "Surprise",
    4: "Fear", 5: "Disgust", 6: "Anger", 7: "Contempt"
}

# -------------------------------
# Model Definitions
# -------------------------------
class ResNetMultiHead(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        base = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        feat_dim = 512
        self.expr_head = nn.Linear(feat_dim, num_classes)
        self.val_head = nn.Linear(feat_dim, 1)
        self.aro_head = nn.Linear(feat_dim, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        return {
            "logits": self.expr_head(feat),
            "valence": self.val_head(feat).squeeze(1),
            "arousal": self.aro_head(feat).squeeze(1)
        }

class VGGMultiHead(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        vgg = models.vgg16(pretrained=True)
        self.backbone = vgg.features
        self.avgpool = vgg.avgpool
        self.fc = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
        )
        self.expr_head = nn.Linear(4096, num_classes)
        self.val_head = nn.Linear(4096, 1)
        self.aro_head = nn.Linear(4096, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return {
            "logits": self.expr_head(x),
            "valence": self.val_head(x).squeeze(1),
            "arousal": self.aro_head(x).squeeze(1)
        }

# -------------------------------
# Preprocessing
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -------------------------------
# Utility Functions
# -------------------------------
def load_and_test_model(model_path, model_type="resnet",
                        images_dir="images", annotations_dir="annotations",
                        num_samples=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if model_type.lower() == "resnet":
        model = ResNetMultiHead()
    elif model_type.lower() == "vgg":
        model = VGGMultiHead()
    else:
        raise ValueError("model_type must be 'resnet' or 'vgg'")

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # Load sample images
    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
    sample_files = random.sample(image_files, min(num_samples, len(image_files)))

    results = []
    with torch.no_grad():
        for img_file in sample_files:
            img_id = os.path.splitext(img_file)[0]
            img_path = os.path.join(images_dir, img_file)
            image = Image.open(img_path).convert("RGB")

            input_tensor = transform(image).unsqueeze(0).to(device)

            # Load ground truth
            true_exp = int(np.load(os.path.join(annotations_dir, f"{img_id}_exp.npy")))
            true_val = float(np.load(os.path.join(annotations_dir, f"{img_id}_val.npy")))
            true_aro = float(np.load(os.path.join(annotations_dir, f"{img_id}_aro.npy")))

            # Inference
            outputs = model(input_tensor)
            logits = outputs["logits"]
            pred_val = outputs["valence"].item()
            pred_aro = outputs["arousal"].item()

            # Predicted emotion
            probabilities = torch.softmax(logits, dim=1)
            confidence, pred_exp = torch.max(probabilities, dim=1)
            pred_exp = pred_exp.item()
            confidence = confidence.item()

            results.append({
                'id': img_id,
                'true_emotion': true_exp,
                'pred_emotion': pred_exp,
                'confidence': confidence,
                'true_valence': true_val,
                'pred_valence': pred_val,
                'true_arousal': true_aro,
                'pred_arousal': pred_aro,
                'correct': (pred_exp == true_exp)
            })
    return results

def print_summary(results, model_name="Model"):
    correct_count = sum(1 for r in results if r['correct'])
    total_count = len(results)
    acc = correct_count / total_count if total_count else 0

    print(f"\n=== {model_name} SUMMARY ===")
    print(f"Samples tested: {total_count}")
    print(f"Correct: {correct_count}, Wrong: {total_count - correct_count}")
    print(f"Accuracy: {acc:.3f} ({acc*100:.1f}%)")

    print("\nSample predictions:")
    for i, r in enumerate(results[:5]):
        status = "✓" if r['correct'] else "✗"
        print(f"{status} {r['id']}: True={EMOTIONS[r['true_emotion']]}, "
              f"Pred={EMOTIONS[r['pred_emotion']]}, Conf={r['confidence']:.3f}")

# -------------------------------
# Main
# -------------------------------
if __name__ == "__main__":
    # Test ResNet18
    if os.path.exists("best_resnet18.pt"):
        resnet_results = load_and_test_model("best_resnet18.pt", "resnet")
        print_summary(resnet_results, "ResNet18")
    else:
        print("ResNet18 checkpoint not found!")

    # Test VGG16
    if os.path.exists("best_vgg.pt"):
        vgg_results = load_and_test_model("best_vgg.pt", "vgg")
        print_summary(vgg_results, "VGG16")
    else:
        print("VGG16 checkpoint not found!")
